# TP 02 Blockchain : Preuve d'enjeu (Proof of Stake)
## Noeud Validateur n°1 : Idriss (Mise : 40 ETH)

Ce notebook est le nœud local du participant **Idriss**. Il contient :
- La configuration de connexion au reseau
- L'implementation locale et autonome des regles du protocole (R2, R6)
- L'enregistrement de la mise (40 ETH)
- Les outils de controle et de verification independante de la blockchain.


## Configuration et Fonctions du Participant


In [1]:
import requests

RESEAU = "http://127.0.0.1:5000"   # Adresse du serveur PoS
NUM = 1                            # Numero du participant
NOM = "Idriss"                     # Prenom du validateur
MISE = 40                          # Mise initiale engagee (>= 10 ETH)

print(f"Participant 1 : Idriss initialise avec une mise de 40 ETH.")


Participant 1 : Idriss initialise avec une mise de 40 ETH.


## Fonctions locales de calcul et de controle (R2, R6)
Ces fonctions s'executent localement chez le participant sans interroger le serveur, lui permettant de verifier la validite des blocs et l'equite du reseau.


In [2]:
def choisir_proposant(mises, tirage):
    """R2, recopiee chez le participant pour controler le reseau.
    
    Parametres : mises (liste de couples (nom, mise)), tirage (float)
    Retour     : str, le nom du proposant attendu
    """
    cumul = 0.0
    for n, m in mises:
        cumul += m
        if tirage <= cumul:
            return n
    return mises[-1][0] if mises else None


def verifier_bloc(bloc):
    """R6 : controle un bloc telecharge, sans rien demander au reseau.
    
    Parametres : bloc (dict : numero, proposant, tirage, mises)
    Retour     : bool, True si le bloc est valide
    """
    return bloc["proposant"] == choisir_proposant(bloc["mises"], bloc["tirage"])


def verifier_chaine(chaine):
    """R6 : applique verifier_bloc a toute la chaine.
    
    Parametres : chaine (liste de dict)
    Retour     : liste des numeros (int) des blocs falsifies
    """
    return [b["numero"] for b in chaine if not verifier_bloc(b)]


def parts(mises):
    """Part de chaque validateur dans la mise totale.
    
    Parametres : mises (liste de couples)
    Retour     : dict {nom (str): pourcentage (float)}
    """
    total = sum(m for _, m in mises)
    if total == 0:
        return {n: 0.0 for n, _ in mises}
    return {n: 100.0 * m / total for n, m in mises}


def afficher_parts(validateurs, chaine):
    """Tableau part de mise / part des blocs, avec barres de '#'.
    
    Parametres : validateurs (liste de dict), chaine (liste de dict)
    Retour     : aucun (affichage)
    """
    total = sum(v["mise"] for v in validateurs)
    print(f"taille de la chaine : {len(chaine)}")
    print(f"{'Validateur':<10} {'Part de mise':>14} {'Part des blocs':>16}")
    for v in validateurs:
        p_m = 100.0 * v["mise"] / total if total > 0 else 0.0
        p_b = 100.0 * v["blocs"] / len(chaine) if len(chaine) > 0 else 0.0
        barre = "#" * int(p_b / 2)
        print(f"{v['nom']:<10} {p_m:>12.1f} % {p_b:>14.1f} %  {barre}")


## Etape 1 : Rejoindre le reseau (R1)
Enregistrement de la mise aupres du serveur Flask.


In [3]:
reponse = requests.post(RESEAU + "/rejoindre", json={"nom": NOM, "mise": MISE})
print("Statut HTTP :", reponse.status_code)
print("Reponse serveur :", reponse.json())


Statut HTTP : 200
Reponse serveur : {'mise_totale': 40, 'validateur': {'blocs': 0, 'exclu': False, 'mise': 40, 'nom': 'Idriss'}}


## Etape 2 : Consultation de l'etat des validateurs (R2)
Verification de la liste des validateurs presents et de la part de mise de chacun.


In [4]:
validateurs = requests.get(RESEAU + "/validateurs").json()
mises = [(v["nom"], v["mise"]) for v in validateurs]
parts_dict = parts(mises)

total_mise = sum(m for _, m in mises)
print(f"Mise totale du reseau : {total_mise:.0f} ETH\n")
for nom_v, mise_v in mises:
    print(f"{nom_v:<10} {mise_v:>5.0f} ETH {parts_dict[nom_v]:>10.1f} %")


Mise totale du reseau : 40 ETH

Idriss        40 ETH      100.0 %


## Etape 3 : Verification locale de la regle de tirage (R2)
Verification que la fonction `choisir_proposant` locale associe correctement les tirages de test aux validateurs.


In [5]:
for t in [52.3, 12.0, 95.5]:
    proposant = choisir_proposant(mises, t)
    print(f"tirage = {t:<4} -> proposant : {proposant}")


tirage = 52.3 -> proposant : Idriss
tirage = 12.0 -> proposant : Idriss
tirage = 95.5 -> proposant : Idriss


## Etape 4 & 5 : Audit de la blockchain (R3, R4, R6)
Telechargement de la chaine locale (`GET /chaine`) et verification de l'authenticite de chaque bloc.


In [6]:
chaine = requests.get(RESEAU + "/chaine").json()
print(f"Nombre de blocs dans la chaine : {len(chaine)}\n")

if chaine:
    print(f"{'Bloc':>4} {'Signe par':<10} {'Tirage':>7} {'Recalcul':<10} {'Verdict'}")
    for b in chaine[:15]:
        recalc = choisir_proposant(b["mises"], b["tirage"])
        verdict = "ok" if verifier_bloc(b) else "FALSIFIE"
        print(f"{b['numero']:>4} {b['proposant']:<10} {b['tirage']:>7.2f} {recalc:<10} {verdict}")

    falsifies = verifier_chaine(chaine)
    print("\nBlocs falsifies detectes localement :", falsifies)
else:
    print("La chaine est actuellement vide.")


Nombre de blocs dans la chaine : 0

La chaine est actuellement vide.
